# Movies Gold Layer

Builds the analytics-ready star schema in `movies_gold`: `fact_movies` (profit & ROI metrics for Power BI reporting) plus the `dim_genre`, `dim_country`, `dim_data_source`, and `dim_time` dimensions. Depends on the **02_silver_layer** notebook.

In [0]:
from pyspark.sql.functions import col, expr, round

# 1. Read the clean data from Silver Delta table
df_silver = spark.read.table("movies_silver.movies")

# 2. Gold Layer Feature Engineering & Analytics Modeling
# Let's create high-value business metrics: Profit and Return on Investment (ROI)
df_gold = (
    df_silver
    # Calculate Net Profit (Revenue minus Budget)
    .withColumn("profit", col("revenue") - col("budget"))
    # Calculate ROI Percentage/Ratio safely (avoiding division by zero)
    .withColumn(
        "roi",
        round(
            expr(
                "CASE WHEN budget > 0 THEN (revenue - budget) / budget ELSE 0"
                " END"
            ),
            2,
        ),
    )
)

# 3. Save as the Gold Delta Table (Optimized for Power BI reporting)
spark.sql("CREATE SCHEMA IF NOT EXISTS movies_gold")

gold_table_name = "movies_gold.fact_movies"

df_gold.write.format("delta").mode("overwrite").saveAsTable(gold_table_name)

print(f"Gold table '{gold_table_name}' successfully created!")

# Preview the new financial metrics
display(
    df_gold.select(
        "title", "budget", "revenue", "profit", "roi", "genres"
    ).limit(10)
)

In [0]:
from pyspark.sql.functions import col, explode, trim, split, monotonically_increasing_id

# 1. Read from Silver table
df_silver = spark.read.table("movies_silver.movies")

# 2. Split the comma-separated genres, explode them into separate rows, and clean up
df_genres = (
    df_silver
    # Split string by comma into an array
    .withColumn("genre_array", split(col("genres"), ","))
    # Explode the array so each genre gets its own row
    .withColumn("genre", explode(col("genre_array")))
    # Trim any accidental whitespace
    .withColumn("genre", trim(col("genre")))
    # Select only the distinct genres
    .select("genre")
    .distinct()
    # Filter out empty or null genres
    .filter((col("genre").isNotNull()) & (col("genre") != ""))
    # Add a unique surrogate key for the dimension table
    .withColumn("genre_id", monotonically_increasing_id())
)

# 3. Save as a Gold Dimension Table
spark.sql("CREATE SCHEMA IF NOT EXISTS movies_gold")

dim_genre_table = "movies_gold.dim_genre"
df_genres.write.format("delta").mode("overwrite").saveAsTable(dim_genre_table)

print(f"Dimension table '{dim_genre_table}' successfully created!")

# Preview the unique genres
display(df_genres.limit(10))

In [0]:
from pyspark.sql.functions import col, trim, monotonically_increasing_id

# 1. Read from Silver table
df_silver = spark.read.table("movies_silver.movies")

# 2. Extract distinct countries from the comma-separated 'countries' column
df_countries = (
    df_silver
    # Filter out rows with null or empty countries
    .filter((col("countries").isNotNull()) & (col("countries") != ""))
    # Select distinct countries directly (values are single, not comma-separated in practice)
    .select("countries")
    .distinct()
    # Clean up whitespace
    .withColumn("country", trim(col("countries")))
    .select("country")
    .distinct()
    .filter((col("country").isNotNull()) & (col("country") != ""))
    # Add a unique surrogate key
    .withColumn("country_id", monotonically_increasing_id())
)

# 3. Save as a Gold Dimension Table
spark.sql("CREATE SCHEMA IF NOT EXISTS movies_gold")

dim_country_table = "movies_gold.dim_country"
df_countries.write.format("delta").mode("overwrite").saveAsTable(dim_country_table)

print(f"Dimension table '{dim_country_table}' successfully created!")

# Preview the unique countries
display(df_countries.limit(10))

In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id

# 1. Read from Silver table
df_silver = spark.read.table("movies_silver.movies")

# 2. Extract distinct data sources
df_sources = (
    df_silver
    .select("data_source")
    .distinct()
    .filter((col("data_source").isNotNull()) & (col("data_source") != ""))
    .withColumn("source_id", monotonically_increasing_id())
)

# 3. Save as a Gold Dimension Table
spark.sql("CREATE SCHEMA IF NOT EXISTS movies_gold")

dim_source_table = "movies_gold.dim_data_source"
df_sources.write.format("delta").mode("overwrite").saveAsTable(dim_source_table)

print(f"Dimension table '{dim_source_table}' successfully created!")

# Preview the data sources
display(df_sources)

In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id

# 1. Read from Silver table
df_silver = spark.read.table("movies_silver.movies")

# 2. Build a Time Dimension from release_year and decade
df_time = (
    df_silver
    .select("release_year", "decade")
    .distinct()
    .filter(col("release_year").isNotNull())
    .withColumn("year", col("release_year").cast("int"))
    .withColumn("decade", col("decade").cast("int"))
    .withColumn("century", (col("release_year") / 100).cast("int") + 1)
    .select("year", "decade", "century")
    .orderBy("year")
    .withColumn("year_id", monotonically_increasing_id())
)

# 3. Save as a Gold Dimension Table
spark.sql("CREATE SCHEMA IF NOT EXISTS movies_gold")

dim_time_table = "movies_gold.dim_time"
df_time.write.format("delta").mode("overwrite").saveAsTable(dim_time_table)

print(f"Dimension table '{dim_time_table}' successfully created!")

# Preview the time dimension
display(df_time.limit(10))